# 11. Randomized Numerical Linear Algebra (RandNLA), Spectral Autograd & Hutch++

### Historical Lineage & Theoretical Foundations
- **Lineage:** *Halko, Martinsson, & Tropp (2011, Finding Structure with Randomness: Probabilistic Algorithms for Constructing Approximate Matrix Decompositions); Meyer, Musco, Musco, & Woodruff (2021, Hutch++: Optimal and Adaptive Stochastic Trace Estimation); Giles (2008, An Extended Collection of Matrix Derivative Results for Forward and Reverse Mode Automatic Differentiation); Hutchinson (1989)*.
- **Core Purpose:** High-performance randomized low-rank sketching, Modified Gram-Schmidt thin QR decomposition with canonical strictly positive phase, optimal $\mathcal{O}(1/\epsilon)$ Hutch++ stochastic trace estimation, and Lorentzian Cauchy-broadened spectral autograd to prevent singular value collision catastrophe.

### Decision Boundary: When NOT to Use
> [!WARNING]
> - **DO NOT compute exact SVD on tall/wide large-scale matrices without sketching:** Full SVD scales as $\mathcal{O}(m n \min(m, n))$, whereas Randomized SVD computes near-optimal low-rank factorizations in $\mathcal{O}(m n (k+p))$ with negligible spectral loss.
> - **DO NOT evaluate trace via explicit matrix diagonalization or standard Hutchinson when $\epsilon < 10^{-2}$:** Classical Hutchinson requires $\mathcal{O}(1/\epsilon^2)$ matrix-vector queries; Hutch++ accelerates convergence to $\mathcal{O}(1/\epsilon)$ queries via low-rank projection.
> - **DO NOT backpropagate through unregularized singular value decompositions when singular values collide:** When $\sigma_i \approx \sigma_j$, the Giles coupling term $F_{ij} = \frac{1}{\sigma_j^2 - \sigma_i^2}$ divides by zero and produces `NaN` gradients; always apply Lorentzian Cauchy broadening.


### 2. Mathematical Foundations & Governing Equations

#### 1. Modified Gram-Schmidt (MGS) QR Decomposition
$$\mathbf{v}_j = \mathbf{y}_j - \sum_{i=0}^{j-1} \langle \mathbf{q}_i, \mathbf{v}_j \rangle \mathbf{q}_i, \quad r_{jj} = \|\mathbf{v}_j\|_2, \quad \mathbf{q}_j = \frac{\mathbf{v}_j}{r_{jj}}$$

#### 2. Hutch++ Stochastic Trace Estimator
$$\hat{\operatorname{tr}}_{\text{Hutch++}}(\mathbf{A}) = \operatorname{tr}(\mathbf{Q}^\top \mathbf{A} \mathbf{Q}) + \frac{1}{m} \sum_{i=1}^m \mathbf{g}_i^\top (\mathbf{A} - \mathbf{Q}\mathbf{Q}^\top \mathbf{A}) \mathbf{g}_i$$

#### 3. Lorentzian-Broadened Spectral Autograd Coupling Matrix
$$\tilde{F}_{ij}^{(\epsilon)} = \begin{cases} \frac{\sigma_j - \sigma_i}{(\sigma_j - \sigma_i)^2 + \epsilon^2} & \text{if } i \neq j \\ 0 & \text{if } i = j \end{cases}$$

**Asymptotic Complexity Analysis:**
- **Randomized Range Finder:** Time $\mathcal{O}(m n l)$ where $l = k + p \ll \min(m, n)$; Space $\mathcal{O}(m l)$.
- **Hutch++ Trace Estimation:** Query complexity $\mathcal{O}(1/\epsilon)$ matrix-vector multiplies to achieve $(1 \pm \epsilon)$ relative error; Space $\mathcal{O}(n l + n m)$.
- **Lorentzian Broadening:** Time $\mathcal{O}(k^2)$ elementwise evaluation; Space $\mathcal{O}(k^2)$ auxiliary gradient storage.


In [ ]:
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn

import core

env = core.setup_env(seed=42)
ckpt_path = Path("weights/00_foundations/11_randomized_numerical_linear_algebra_spectral_autograd_and_hutchpp.safetensors")
if not ckpt_path.is_file():
    ckpt_path = Path("../../weights/00_foundations/11_randomized_numerical_linear_algebra_spectral_autograd_and_hutchpp.safetensors")
weights, test_in, golden_out, meta = core.parity.load_reference_checkpoint(ckpt_path)

batch_size = int(meta.get("batch_size", 8))
in_dim = int(meta.get("in_dim", 6))
out_dim = int(meta.get("out_dim", 4))
matrix_a = weights["matrix_a"]
s_sketch = weights["s_sketch"]
g_rademacher = weights["g_rademacher"]
exact_trace = float(weights["exact_trace"][0].item())

print(f"Device: {env.device} | Batch: {batch_size} | In: {in_dim} | Out: {out_dim} | Checkpoint: {ckpt_path.name}")


In [ ]:
def modified_gram_schmidt_qr(y: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    m, l = y.shape
    q = y.clone()
    r = torch.zeros(l, l, dtype=y.dtype, device=y.device)
    for j in range(l):
        v = q[:, j].clone()
        for i in range(j):
            rij = torch.dot(q[:, i], v)
            r[i, j] = rij
            v = v - rij * q[:, i]
        rjj = torch.linalg.norm(v)
        r[j, j] = rjj
        q[:, j] = v / (rjj + 1e-12)
    return q, r

def hutch_plus_plus_trace(a: torch.Tensor, s_sketch: torch.Tensor, g_mc: torch.Tensor) -> torch.Tensor:
    y = a @ s_sketch
    q, _ = modified_gram_schmidt_qr(y)
    tr_proj = torch.trace(q.T @ a @ q)
    a_g = a @ g_mc
    residual_g = a_g - q @ (q.T @ a_g)
    m = float(g_mc.shape[1])
    tr_res = torch.sum(torch.sum(g_mc * residual_g, dim=0)) / m
    return tr_proj + tr_res

def lorentzian_broadened_f_matrix(singular_values: torch.Tensor, epsilon: float = 1e-2) -> torch.Tensor:
    n = singular_values.shape[0]
    delta = singular_values.unsqueeze(0) - singular_values.unsqueeze(1)
    f = delta / (delta**2 + (epsilon**2))
    return f * (1.0 - torch.eye(n, dtype=f.dtype, device=f.device))

def power_iteration_spectral_norm(w: torch.Tensor, v_init: torch.Tensor, num_iters: int = 5) -> torch.Tensor:
    v = v_init / (torch.linalg.norm(v_init) + 1e-12)
    u = torch.zeros(w.shape[0], dtype=w.dtype, device=w.device)
    for _ in range(num_iters):
        u = w @ v
        u = u / (torch.linalg.norm(u) + 1e-12)
        v = w.T @ u
        v = v / (torch.linalg.norm(v) + 1e-12)
    return torch.dot(u, w @ v)

class SpectralLinear(nn.Module):
    def __init__(self, in_features: int, out_features: int) -> None:
        super().__init__()
        self.weight = nn.Parameter(torch.empty(in_features, out_features))
        self.bias = nn.Parameter(torch.empty(out_features))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x @ self.weight + self.bias

model = SpectralLinear(in_dim, out_dim)
with torch.no_grad():
    model.weight.copy_(weights["weight"])
    model.bias.copy_(weights["bias"])
print("SpectralLinear and RandNLA operators initialized.")


In [ ]:
# 1. MGS-QR Parity
y_sketch = matrix_a @ s_sketch
q_pred, r_pred = modified_gram_schmidt_qr(y_sketch)
torch.testing.assert_close(q_pred, weights["q_basis"], atol=1e-5, rtol=1e-5)
torch.testing.assert_close(r_pred, weights["r_basis"], atol=1e-5, rtol=1e-5)

# 2. Hutch++ Stochastic Trace Parity
hutch_est = hutch_plus_plus_trace(matrix_a, s_sketch, g_rademacher)
torch.testing.assert_close(hutch_est.unsqueeze(0), weights["hutch_trace_estimate"], atol=1e-5, rtol=1e-5)

# 3. Power Iteration Spectral Norm Parity
sigma_pred = power_iteration_spectral_norm(weights["weight"], weights["v_init"], num_iters=5)
torch.testing.assert_close(sigma_pred.unsqueeze(0), weights["spectral_sigma"], atol=1e-5, rtol=1e-5)

# 4. Lorentzian Coupling Matrix Parity
f_pred = lorentzian_broadened_f_matrix(weights["colliding_sigma"], epsilon=1e-2)
torch.testing.assert_close(f_pred, weights["lorentzian_f"], atol=1e-5, rtol=1e-5)

# 5. Forward Prediction Parity
pred_out = model(test_in)
torch.testing.assert_close(pred_out, golden_out, atol=1e-5, rtol=1e-5)
print("Transitive Parity Verified across all 5 operators (atol < 1e-5).")


In [ ]:
def run_hutch():
    return hutch_plus_plus_trace(matrix_a, s_sketch, g_rademacher)

def run_power_iteration():
    return power_iteration_spectral_norm(weights["weight"], weights["v_init"], num_iters=5)

def run_forward():
    return model(test_in)

track_fns = {"Hutch++ Trace": run_hutch, "Power Iteration": run_power_iteration, "SpectralLinear Fwd": run_forward}
bench_results = core.benchmark.compare_tracks(track_fns, warmup=3, iters=15, batch_size=batch_size)
print(core.benchmark.format_benchmark_table(bench_results))


In [ ]:
# [!WARNING] PATHOLOGICAL MISUSE & FAILURE MODE: Singular Value Degeneracy & Autograd Explosion
# 1. Flawed Anti-Pattern: Unbroadened Giles SVD autograd with colliding singular values
colliding_sigmas = weights["colliding_sigma"]
delta = colliding_sigmas.unsqueeze(0) - colliding_sigmas.unsqueeze(1)

with np.errstate(divide="ignore", invalid="ignore"):
    naive_f = 1.0 / delta

has_nan_or_inf = bool(torch.isnan(naive_f).any() or torch.isinf(naive_f).any())
assert has_nan_or_inf, "Naive SVD autograd on colliding singular values must produce NaN or Inf"
print(f"[FAILURE DEMONSTRATED] Naive Giles SVD autograd divided by zero on colliding singular values: F[1, 2] = {naive_f[1, 2].item()}")

# 2. Correct Pattern: Lorentzian-broadened Cauchy coupling matrix is completely finite
broadened_f = lorentzian_broadened_f_matrix(colliding_sigmas, epsilon=1e-2)
assert torch.isfinite(broadened_f).all(), "Lorentzian broadened matrix must be completely finite"
print(f"[CORRECT] Lorentzian Cauchy broadening regularized collision smoothly: F[1, 2] = {broadened_f[1, 2].item():.4f}")

# 3. Compilation Verification
ok_exp, msg_exp = core.export.verify_torch_export(model, test_in)
ok_comp, msg_comp = core.export.verify_torch_compile_fullgraph(model, test_in)
assert ok_exp and ok_comp, "Compilation / export failed"
print("[CORRECT] Model verified with torch.compile(fullgraph=True) and exported.")


## Architectural Topology, Key Takeaways & Curriculum Navigation

### RandNLA & Spectral Autograd Topology
<svg width="680" height="180" xmlns="http://www.w3.org/2000/svg" style="background:#0f172a; border-radius:8px; font-family:system-ui, sans-serif;">
  <defs>
    <marker id="arrow-s" viewBox="0 0 10 10" refX="6" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
      <path d="M 0 1 L 10 5 L 0 9 z" fill="#38bdf8"/>
    </marker>
  </defs>
  <text x="30" y="25" fill="#94a3b8" font-size="12" font-weight="bold">RANDOMIZED LINEAR ALGEBRA &amp; SPECTRAL AUTOGRAD</text>
  <rect x="30" y="50" width="150" height="35" rx="6" fill="#1e293b" stroke="#38bdf8" stroke-width="1.5"/>
  <text x="105" y="72" fill="#38bdf8" font-size="11" text-anchor="middle">Matrix A + Sketch S</text>
  <line x1="180" y1="67" x2="240" y2="67" stroke="#38bdf8" stroke-width="2" marker-end="url(#arrow-s)"/>
  <rect x="240" y="50" width="180" height="35" rx="6" fill="#1e293b" stroke="#10b981" stroke-width="1.5"/>
  <text x="330" y="72" fill="#10b981" font-size="11" text-anchor="middle">MGS-QR Orthonormal Q</text>
  <line x1="420" y1="67" x2="480" y2="67" stroke="#38bdf8" stroke-width="2" marker-end="url(#arrow-s)"/>
  <rect x="480" y="50" width="170" height="35" rx="6" fill="#1e293b" stroke="#a855f7" stroke-width="1.5"/>
  <text x="565" y="72" fill="#a855f7" font-size="11" text-anchor="middle">Hutch++ Trace Est.</text>
  <rect x="30" y="110" width="620" height="40" rx="6" fill="#0f172a" stroke="#f59e0b" stroke-width="1.5" stroke-dasharray="4 4"/>
  <text x="340" y="135" fill="#f59e0b" font-size="11" text-anchor="middle">Lorentzian Broadening: Skew Coupling (Delta / (Delta^2 + eps^2)) Prevents NaN Blowup</text>
</svg>

### Key Takeaways
1. **Query Efficiency:** Hutch++ combines low-rank sketching with Monte Carlo to achieve $\mathcal{O}(1/\epsilon)$ query complexity vs $\mathcal{O}(1/\epsilon^2)$ for standard Hutchinson.
2. **Canonical Phase Invariance:** Modified Gram-Schmidt QR guarantees strictly positive diagonal $R_{jj} > 0$, eliminating arbitrary basis sign flips across backends.
3. **Spectral Autograd Safety:** Always apply Lorentzian Cauchy broadening to off-diagonal singular value differences to eliminate division-by-zero during backpropagation.

---
**Curriculum Navigation:**  
&larr; [Previous: 10. Floating-Point Arithmetic, Conditioning & Numerical Stability](10_floating_point_arithmetic_conditioning_and_numerical_stability.ipynb) | [Master Index: README.md](../../README.md) | Next: Category 01: Core Architecture Foundations (01_mlp_residual_bottlenecks_and_activations) (Upcoming in Stage 1) &rarr;
